## はじめに

このノートブックでは、Part 1 で作成した raw 層のテキストを Cortex AI 関数で加工し、分析や検索にそのまま使える **conform 層**のテーブルを作成します。通話や PDF の中にしかなかった情報を列として取り出し、注文データと組み合わせて集計できることを体験します。

**前提条件:**
- `part1_20261216.ipynb` が完了していること（`CALL_TRANSCRIPTS`、`DOC_PARSED` などが作成済み）

**主な処理内容:**
1. AI 関数を試す（AI_EXTRACT, AI_REDACT）
2. conform 層のテーブルを作成する（通話の分析、PDF の属性抽出）
3. 取り出した列を使って集計する
4. 検索用のチャンクを作成する（Part 3 の Cortex Search で使用）

**生成されるテーブル（conform 層）:**
- `CALL_ANALYSIS`: 通話ごとの問い合わせ種別・商品カテゴリ・感情・解約リスク・要約と、個人情報をマスクした本文（98件）
- `DOC_ATTRIBUTES`: PDF ごとのタイトル・文書の種類・部署・公開範囲・発行日・対象商品（11件）
- `KNOWLEDGE_CHUNKS`: 通話と PDF を検索用に分割し、属性を付けたチャンク（200件）

> **料金について**: AI 関数は処理量に応じてクレジットを消費します。実行前に[最新の料金](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost)を確認してください。

> **途中で遅れた場合**: `checkpoint_20261216.sql` の「Part 1」で開始時点、「Part 2」で完了時点のテーブルを復元できます。

In [ ]:
%%sql -r result_env_setup
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するロール、ウェアハウス、スキーマを設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/part2/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(15, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 1. Cortex AI 関数を試してみよう

テーブル全体を処理する前に、数件だけで AI 関数の動きを確かめます。

### 1-1. AI_EXTRACT — 情報抽出

テキストから、指定した項目を取り出します。取り出したい項目と説明を `responseFormat` に書くだけで、結果が JSON で返ります。

In [ ]:
display_image('ai_extract.png')

In [ ]:
%%sql -r result_try_extract
-- ============================================================================
-- AI_EXTRACT: 通話から問い合わせ種別を取り出してみよう
-- ============================================================================
--   inquiry_free  : '問い合わせ種別' → AI が自由に表現する（「返品」「配送状況の確認」など）
--   inquiry_choice: 選択肢を書く     → 指定した中から選ばれ、集計しやすくなる
SELECT
    call_id,
    LEFT(transcript_text, 60) AS transcript_preview,
    AI_EXTRACT(
        text => transcript_text,
        responseFormat => {'inquiry_type': '問い合わせ種別'}
    ):response:inquiry_type::VARCHAR AS inquiry_free,
    AI_EXTRACT(
        text => transcript_text,
        responseFormat => {'inquiry_type': '問い合わせ種別を次から1つ: 商品不具合, 配送遅延, 返品・交換, 注文キャンセル, 商品問い合わせ, 支払い, 会員・ポイント, その他'}
    ):response:inquiry_type::VARCHAR AS inquiry_choice
FROM CALL_TRANSCRIPTS
WHERE source = '文字起こし済み'
ORDER BY call_id
LIMIT 5;

### 1-2. AI_REDACT — 個人情報のマスク

通話には氏名や電話番号などの個人情報が含まれます。`AI_REDACT` で `[NAME]` のような記号に置き換えてから、検索や共有に使います。

In [ ]:
display_image('ai_redact.png')

In [ ]:
%%sql -r result_try_redact
-- ============================================================================
-- AI_REDACT: 通話の個人情報をマスクしてみよう
-- ============================================================================
SELECT
    call_id,
    LEFT(transcript_text, 120) AS original_text,
    LEFT(AI_REDACT(transcript_text)::VARCHAR, 120) AS masked_text
FROM CALL_TRANSCRIPTS
WHERE source = '文字起こし済み'
ORDER BY call_id
LIMIT 3;

## 2. conform 層のテーブルの作成

セクション1で試した AI 関数を全件に適用し、conform 層のテーブルを作成します。

### 2-1. 通話の分析（CALL_ANALYSIS）

1回の `AI_EXTRACT` で、問い合わせ種別・商品名・商品カテゴリ・感情・解約リスク・要約の6項目をまとめて取り出します。あわせて `AI_REDACT` で本文の個人情報をマスクします。

AI は選択肢にない値（商品名など）を返すことがあります。集計や検索の絞り込みで困らないよう、商品カテゴリは選択肢以外を空にする後処理を SQL で行います。

> **ヒント**: このセクションの CREATE TABLE は98件を処理するため、実行に数分かかることがあります。

In [ ]:
%%sql -r result_call_analysis
-- ============================================================================
-- 通話の AI 分析と conform 層への保存
-- ============================================================================
CREATE OR REPLACE TABLE CALL_ANALYSIS AS
WITH extracted AS (
    SELECT
        t.*,
        AI_EXTRACT(
            text => transcript_text,
            responseFormat => {
                'inquiry_type': '問い合わせ種別を次から1つ: 商品不具合, 配送遅延, 返品・交換, 注文キャンセル, 商品問い合わせ, 支払い, 会員・ポイント, その他',
                'product_name': '話題になっている商品名。なければ空',
                'product_category': '話題になっている商品のカテゴリを次から1つ: 照明, 家具, 収納, 調理器具, 食器, 文具, トラベル, ギフト, ファッション小物, ルームウェア。商品の話でなければ空',
                'sentiment': '顧客の感情を次から1つ: ネガティブ, 中立, ポジティブ',
                'churn_risk': '顧客が他社に乗り換えたり利用をやめたりする可能性を次から1つ: 高, 中, 低',
                'summary': '通話内容の要約（60文字以内）'
            }
        ):response AS r
    FROM CALL_TRANSCRIPTS t
)
SELECT
    call_id, call_datetime, customer_id, order_id, product_id, source,
    r:inquiry_type::VARCHAR AS inquiry_type,
    r:product_name::VARCHAR AS product_name,
    -- 選択肢にない値は空にする（AI が商品名などを返すことがあるため）
    IFF(r:product_category::VARCHAR IN ('照明', '家具', '収納', '調理器具', '食器', '文具', 'トラベル', 'ギフト', 'ファッション小物', 'ルームウェア'),
        r:product_category::VARCHAR, NULL) AS product_category,
    r:sentiment::VARCHAR AS sentiment,
    r:churn_risk::VARCHAR AS churn_risk,
    r:summary::VARCHAR AS summary,
    -- AI_REDACT の結果は VARIANT で返るため、文字列に変換して保存
    AI_REDACT(transcript_text)::VARCHAR AS transcript_masked
FROM extracted;

-- 問い合わせ種別ごとの件数を確認
SELECT inquiry_type, COUNT(*) AS call_count
FROM CALL_ANALYSIS
GROUP BY inquiry_type
ORDER BY call_count DESC;

### 2-2. PDF の属性抽出（DOC_ATTRIBUTES）

PDF の1ページ目（表紙・文書情報）から、タイトル・文書の種類・部署・公開範囲・発行日・対象商品を取り出します。これらは Part 3 の Cortex Search で「品質管理部の文書だけ」「社外公開の文書だけ」のように検索を絞り込む属性になります。

特定の商品を扱わない文書では、AI が「None」などの余計な値を返すことがあります。商品ID（`PRD-` で始まる値）が取れた文書だけに商品カテゴリと商品IDを付けます。

In [ ]:
%%sql -r result_doc_attributes
-- ============================================================================
-- PDF の1ページ目から属性を取り出し、conform 層へ保存
-- ============================================================================
CREATE OR REPLACE TABLE DOC_ATTRIBUTES AS
WITH extracted AS (
    SELECT
        file_name,
        AI_EXTRACT(
            text => page_text,
            responseFormat => {
                'title': '文書のタイトル',
                'doc_type': '文書の種類を次から1つ: FAQ, マニュアル, 品質報告, 通達, 店舗報告',
                'department': '作成した部署を次から1つ: カスタマーサポート部, 品質管理部, 店舗運営部',
                'audience': '公開範囲を次から1つ: 社外公開, 社内限定。お客様向けの文書は社外公開',
                'doc_date': '発行日または最終更新日（YYYY-MM-DD）',
                'product_category': '対象の商品カテゴリを次から1つ: 照明, 家具, 収納, 調理器具, 食器, 文具, トラベル, ギフト, ファッション小物, ルームウェア。特定の商品についての文書でなければ空',
                'product_ids': '対象の商品IDをすべて（例: PRD-LGT-0001）'
            }
        ):response AS r
    FROM DOC_PARSED
    WHERE page_no = 1
),
-- 商品ID（PRD- で始まる値）だけを残す
cleaned AS (
    SELECT
        file_name, r,
        ARRAY_COMPACT(TRANSFORM(
            COALESCE(TRY_PARSE_JSON(r:product_ids::VARCHAR), ARRAY_CONSTRUCT(r:product_ids)),
            x -> IFF(STARTSWITH(x::VARCHAR, 'PRD-'), x::VARCHAR, NULL)
        )) AS ids
    FROM extracted
)
SELECT
    file_name,
    r:title::VARCHAR AS title,
    r:doc_type::VARCHAR AS doc_type,
    r:department::VARCHAR AS department,
    r:audience::VARCHAR AS audience,
    TRY_TO_DATE(r:doc_date::VARCHAR) AS doc_date,
    IFF(ARRAY_SIZE(ids) > 0, r:product_category::VARCHAR, NULL) AS product_category,
    ids AS product_ids
FROM cleaned;

-- 抽出結果を確認
SELECT file_name, title, doc_type, department, audience, doc_date, product_category, product_ids
FROM DOC_ATTRIBUTES
ORDER BY file_name;

## 3. 取り出した列で集計してみよう

通話の中にしかなかった「問い合わせ種別」「感情」「解約リスク」が列になったので、通常のデータと同じように集計できます。さらに `order_id` で注文データとつなぐと、通話と注文の両方から同じ問題が見えてきます。

### 3-1. 月別の問い合わせ件数

In [ ]:
%%sql -r result_monthly_calls
-- ============================================================================
-- 月別の通話件数（音声のサンプル10件は日時がないため除外）
-- ============================================================================
SELECT
    TO_CHAR(DATE_TRUNC('month', call_datetime), 'YYYY-MM') AS call_month,
    COUNT(*) AS call_count,
    COUNT_IF(inquiry_type = '商品不具合') AS defect_calls,
    COUNT_IF(inquiry_type = '配送遅延') AS delay_calls,
    COUNT_IF(sentiment = 'ネガティブ') AS negative_calls
FROM CALL_ANALYSIS
WHERE call_datetime IS NOT NULL
GROUP BY call_month
ORDER BY call_month;

10月から「商品不具合」の通話が急に増えています。どの商品の、どのロットで起きているのかを、注文データとつないで確かめます。

### 3-2. 不具合の通話を注文データとつなぐ

In [ ]:
%%sql -r result_defect_by_lot
-- ============================================================================
-- 「商品不具合」の通話を、商品と入荷ロット別に集計
-- ============================================================================
SELECT
    p.product_name,
    COALESCE(o.lot_no, '（ロット記録なし）') AS lot_no,
    COUNT(*) AS defect_calls,
    COUNT_IF(a.churn_risk = '高') AS high_churn_risk_calls
FROM CALL_ANALYSIS a
JOIN ORDERS o ON o.order_id = a.order_id
JOIN PRODUCTS p ON p.product_id = o.product_id
WHERE a.inquiry_type = '商品不具合'
GROUP BY p.product_name, lot_no
ORDER BY defect_calls DESC;

In [ ]:
%%sql -r result_return_rate
-- ============================================================================
-- 注文データだけで見た、LEDデスクライト 調光式の月別返品率
-- ============================================================================
SELECT
    TO_CHAR(DATE_TRUNC('month', o.order_datetime), 'YYYY-MM') AS order_month,
    COALESCE(o.lot_no, '（ロット記録なし）') AS lot_no,
    COUNT(*) AS order_count,
    ROUND(COUNT_IF(o.order_status = 'returned') / COUNT(*) * 100, 1) AS return_rate_pct
FROM ORDERS o
WHERE o.product_id = 'PRD-LGT-0001'
GROUP BY order_month, lot_no
ORDER BY order_month, lot_no;

通話から取り出した「商品不具合」はロット `LOT-2609-B` に集中しており、注文データでもこのロットの返品率が10月以降に大きく上がっています。通話という非構造化データが、注文データと同じように分析に使えるようになりました。

原因や対応方針は PDF の品質報告書や通達に書かれています。Part 3 以降では、これらの文書を検索できるようにし、AI エージェントが数値と文書の両方から回答できるようにします。

## 4. 検索用チャンクの作成

Cortex Search で検索する単位（チャンク）を作ります。通話は1件を1チャンクにし、マスク済みの本文の先頭に要約を付けます。PDF はページ本文を約800文字ずつに分割します。どちらにも、セクション2で取り出した属性（文書の種類・部署・公開範囲・日付・商品カテゴリなど）を付けます。

In [ ]:
%%sql -r result_knowledge_chunks
-- ============================================================================
-- 通話と PDF を統合した検索用チャンクテーブル
-- ============================================================================
-- 通話の種類・部署・公開範囲は、コールセンターの通話という出どころで決まる
CREATE OR REPLACE TABLE KNOWLEDGE_CHUNKS AS
-- 通話（1通話 = 1チャンク）
SELECT
    call_id AS chunk_id,
    '【要約】' || summary || '\n' || transcript_masked AS chunk_text,
    inquiry_type || COALESCE(' / ' || product_name, '') AS title,
    '通話' AS doc_type,
    'カスタマーサポート部' AS department,
    '社内限定' AS audience,
    call_datetime::DATE AS doc_date,
    product_category,
    IFF(product_id IS NULL, ARRAY_CONSTRUCT(), ARRAY_CONSTRUCT(product_id)) AS product_ids,
    sentiment
FROM CALL_ANALYSIS
UNION ALL
-- PDF（ページ本文を分割し、文書の属性を付ける）
SELECT
    d.file_name || '-p' || d.page_no || '-' || c.index AS chunk_id,
    c.value::VARCHAR AS chunk_text,
    a.title, a.doc_type, a.department, a.audience, a.doc_date,
    a.product_category, a.product_ids,
    NULL AS sentiment
FROM DOC_PARSED d
JOIN DOC_ATTRIBUTES a ON a.file_name = d.file_name,
LATERAL FLATTEN(input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(d.page_text, 'markdown', 800, 100)) c;

-- 文書の種類と公開範囲ごとのチャンク数を確認
SELECT doc_type, audience, COUNT(*) AS chunk_count
FROM KNOWLEDGE_CHUNKS
GROUP BY doc_type, audience
ORDER BY doc_type;

## 5. データ確認

作成した conform 層のテーブルの件数を確認します。

In [ ]:
%%sql -r result_verify
-- ============================================================================
-- conform 層のテーブルのレコード数を確認
-- ============================================================================
SELECT 'CALL_ANALYSIS' AS table_name, COUNT(*) AS record_count, '通話の分析結果（98件）' AS description FROM CALL_ANALYSIS
UNION ALL
SELECT 'DOC_ATTRIBUTES', COUNT(*), 'PDF の属性（11件）' FROM DOC_ATTRIBUTES
UNION ALL
SELECT 'KNOWLEDGE_CHUNKS', COUNT(*), '検索用チャンク（200件）' FROM KNOWLEDGE_CHUNKS;

## まとめ

このノートブックでは、raw 層のテキストを Cortex AI 関数で加工し、conform 層のテーブルを作成しました。

### 作成したテーブル一覧（conform 層）
- `CALL_ANALYSIS`: 通話の問い合わせ種別・商品カテゴリ・感情・解約リスク・要約と、マスク済みの本文
- `DOC_ATTRIBUTES`: PDF のタイトル・文書の種類・部署・公開範囲・発行日・対象商品
- `KNOWLEDGE_CHUNKS`: 属性付きの検索用チャンク

### 使用した Cortex AI 関数
- `AI_EXTRACT`: テキストからの情報抽出（選択肢を指定すると結果がそろう）
- `AI_REDACT`: 個人情報のマスク
- `SPLIT_TEXT_RECURSIVE_CHARACTER`: 検索用のチャンク分割

### わかったこと
- 10月以降、「商品不具合」の通話が急増している
- 不具合の通話は LEDデスクライト 調光式のロット `LOT-2609-B` に集中し、注文データでも同じロットの返品率が上がっている

### 次のステップ
- **Part 3**: `KNOWLEDGE_CHUNKS` から Cortex Search サービスを作成し、属性で絞り込んで検索します